In [ ]:
# Cell 1: imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 130
plt.rcParams["font.size"] = 10

In [ ]:
# Cell 2: file paths and configuration

BASE = Path(r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients")

MODEL_PATH = BASE / "florida_model_df.parquet"
MASTER_PATH = BASE / "florida_event_master.parquet"
COVERAGE_PATH = BASE / "florida_coverage.parquet"
EVAL_PATH = BASE / "florida_rolling_persistence_eval.parquet"

# plotting config
X_MAX_HOURS = 12 * 24      # cap x-axis at ~12 days
COVERAGE_THRESHOLD = 0.30  # 30 percent threshold; auto-handled if customersTracked is in 0-100 scale
SMOOTH_WINDOW = None       # e.g. 3, 6, 12; or None for no smoothing
USE_INTERPOLATION = True   # if False, low-coverage points become gaps
ANIMATE_STEP_HOURS = 3     # frame spacing for GIF
MAX_EVENTS_TO_PLOT = 40    # keep this moderate for readability
RANDOM_STATE = 7

In [ ]:
# Cell 3: load and normalise data

df = pd.read_parquet(MODEL_PATH)

# Parse time fields defensively
for col in ["datetime", "event_start"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Identify county key
county_key = None
for candidate in ["CountyFIPS", "geoid", "county_fips", "countyFips"]:
    if candidate in df.columns:
        county_key = candidate
        break

if county_key is None:
    raise ValueError("Could not find a county key column. Expected CountyFIPS or geoid.")

# Keep only the columns we need, but do not fail if some are missing
needed = [
    "event_id", "storm", county_key, "county", "datetime", "event_start",
    "duration_hours", "gust_mps", "wind_speed_mps", "precip_mm",
    "pressure_hpa", "temp_c", "outageFraction", "customersTracked"
]
keep = [c for c in needed if c in df.columns]
df = df[keep].copy()

# Ensure county codes stay as strings with leading zeros preserved
df[county_key] = df[county_key].astype("string")

# Sort and compute relative hour from event start
df = df.sort_values(["event_id", county_key, "datetime"]).reset_index(drop=True)
df["rel_hour"] = ((df["datetime"] - df["event_start"]).dt.total_seconds() / 3600.0).round().astype("Int64")

# Remove obviously invalid rows
df = df[df["outageFraction"].notna()].copy()
df = df[(df["rel_hour"].notna()) & (df["rel_hour"] >= 0)].copy()

print(df.shape)
print(df[["event_id", "storm", county_key, "county"]].drop_duplicates().shape)

In [ ]:
# Cell 4: helpers

def choose_event_subset(
    df,
    max_events=MAX_EVENTS_TO_PLOT,
    storms=None,
    event_ids=None,
    random_state=RANDOM_STATE,
):
    x = df.copy()
    if storms is not None:
        storms = set(storms)
        x = x[x["storm"].isin(storms)]
    if event_ids is not None:
        event_ids = set(event_ids)
        x = x[x["event_id"].isin(event_ids)]

    event_order = (
        x[["event_id", "event_start"]]
        .drop_duplicates()
        .sort_values("event_start")
    )

    if len(event_order) > max_events:
        # keep a spread over time rather than a random sample
        idx = np.linspace(0, len(event_order) - 1, max_events).round().astype(int)
        event_order = event_order.iloc[idx]

    keep_ids = event_order["event_id"].tolist()
    x = x[x["event_id"].isin(keep_ids)].copy()

    # order events by start time so later events plot darker and on top
    event_rank = {eid: i for i, eid in enumerate(event_order["event_id"].tolist())}
    x["event_rank"] = x["event_id"].map(event_rank)

    return x, event_order

def standardize_customers_tracked(s):
    """
    Returns a series that can be interpreted as either [0,1] fraction or [0,100] percent.
    """
    s = pd.to_numeric(s, errors="coerce")
    if s.dropna().max() is None:
        return s
    # if values look like percentages, threshold will be scaled too
    return s

def coverage_ok_mask(customers_tracked, threshold=0.30):
    x = pd.to_numeric(customers_tracked, errors="coerce")
    finite = x.dropna()
    if len(finite) == 0:
        return pd.Series(False, index=x.index)

    # Auto-handle 0-1 vs 0-100 scale
    if finite.max() > 1.5:
        thr = threshold * 100.0
    else:
        thr = threshold
    return x >= thr

def prep_event_series(
    g,
    value_col="outageFraction",
    coverage_col="customersTracked",
    coverage_threshold=COVERAGE_THRESHOLD,
    x_max_hours=X_MAX_HOURS,
    smooth_window=None,
    interpolate_missing=False,
):
    """
    Prepare a single event-window series on an hourly grid from hour 0 to x_max_hours.
    Returns a dataframe with:
      hour, value, observed, lowcov_removed, interpolated
    """
    # One county-storm-event window. If multiple counties per event_id exist, the caller should subset first.
    g = g.sort_values("datetime").copy()

    # Build hourly axis from 0 to max available hour
    max_hour = g["rel_hour"].max()
    if pd.isna(max_hour):
        return None

    end_hour = int(min(max_hour, x_max_hours))
    full_hours = pd.Index(np.arange(0, end_hour + 1), name="hour")

    # collapse duplicates safely
    s = (
        g.groupby("rel_hour", as_index=True)[value_col]
        .mean()
        .reindex(full_hours)
    )

    cov = (
        g.groupby("rel_hour", as_index=True)[coverage_col]
        .mean()
        .reindex(full_hours)
    )

    observed = s.notna()
    cov_ok = coverage_ok_mask(cov, threshold=coverage_threshold)

    # Remove low coverage points
    filtered = s.where(cov_ok)

    # Optional interpolation for gaps created by low coverage
    if interpolate_missing:
        filled = filtered.interpolate(method="linear", limit_area="inside")
    else:
        filled = filtered

    # Optional smoothing
    if smooth_window is not None and smooth_window > 1:
        filled = filled.rolling(smooth_window, min_periods=1, center=True).mean()

    out = pd.DataFrame({
        "hour": full_hours,
        "value": filled.values,
        "observed": observed.values,
        "cov_ok": cov_ok.values,
        "raw": s.values,
        "filtered": filtered.values,
    })
    out["lowcov_removed"] = out["raw"].notna() & (~out["cov_ok"])
    out["interpolated"] = out["filtered"].isna() & out["value"].notna()

    return out

def split_segments_for_style(x, y, interpolated_mask):
    """
    Split a line into solid and dashed segments.
    A segment is dashed if either endpoint is interpolated.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    interp = np.asarray(interpolated_mask, dtype=bool)

    solid_segments = []
    dashed_segments = []

    for i in range(len(x) - 1):
        if np.any(np.isnan([x[i], x[i+1], y[i], y[i+1]])):
            continue
        seg = [(x[i], y[i]), (x[i+1], y[i+1])]
        if interp[i] or interp[i+1]:
            dashed_segments.append(seg)
        else:
            solid_segments.append(seg)

    return solid_segments, dashed_segments

def add_piecewise_line(ax, x, y, interpolated_mask, color="k", lw=1.4, alpha=0.9, zorder=2):
    solid_segments, dashed_segments = split_segments_for_style(x, y, interpolated_mask)

    if solid_segments:
        lc = LineCollection(solid_segments, colors=[color], linewidths=lw, alpha=alpha, zorder=zorder)
        ax.add_collection(lc)
    if dashed_segments:
        lc = LineCollection(
            dashed_segments,
            colors=[color],
            linewidths=lw,
            linestyles="dashed",
            alpha=alpha,
            zorder=zorder,
        )
        ax.add_collection(lc)

def plot_overlay(
    df,
    value_col="outageFraction",
    wind_col=None,
    coverage_threshold=COVERAGE_THRESHOLD,
    smooth_window=None,
    interpolate_missing=False,
    max_events=MAX_EVENTS_TO_PLOT,
    storms=None,
    event_ids=None,
    x_max_hours=X_MAX_HOURS,
    title=None,
    outfile=None,
):
    x, event_order = choose_event_subset(
        df, max_events=max_events, storms=storms, event_ids=event_ids
    )

    # Prepare per-event series
    prepared = []
    for eid, g in x.groupby("event_id", sort=False):
        series = prep_event_series(
            g,
            value_col=value_col,
            coverage_col="customersTracked",
            coverage_threshold=coverage_threshold,
            x_max_hours=x_max_hours,
            smooth_window=smooth_window,
            interpolate_missing=interpolate_missing,
        )
        if series is not None:
            series["event_id"] = eid
            series["storm"] = g["storm"].iloc[0] if "storm" in g.columns else ""
            series["event_start"] = g["event_start"].iloc[0]
            series["event_rank"] = g["event_rank"].iloc[0]
            prepared.append(series)

    if not prepared:
        raise ValueError("No events available after filtering.")

    # Grayscale: earlier = lighter, later = darker
    n = len(prepared)
    shades = np.linspace(0.75, 0.05, n)  # 0=white, 1=black in Greys
    cmap = mpl.cm.Greys

    if wind_col is not None:
        fig, axes = plt.subplots(
            2, 1, figsize=(12, 8), sharex=True,
            gridspec_kw={"height_ratios": [2, 1], "hspace": 0.08}
        )
        ax0, ax1 = axes
    else:
        fig, ax0 = plt.subplots(figsize=(12, 5.5))
        ax1 = None

    # outage panel
    for i, series in enumerate(sorted(prepared, key=lambda d: d["event_rank"].iloc[0])):
        color = cmap(shades[i])
        xh = series["hour"].to_numpy()
        yh = series["value"].to_numpy()
        im = series["interpolated"].to_numpy()
        add_piecewise_line(ax0, xh, yh, im, color=color, lw=1.5, alpha=0.95, zorder=1 + i)

    ax0.set_ylim(0, 1.0)
    ax0.set_xlim(0, x_max_hours)
    ax0.set_ylabel("outageFraction")
    ax0.grid(True, alpha=0.25)
    ax0.set_title(title or f"{value_col} trajectories across events")

    # optional wind panel
    if ax1 is not None:
        for i, series in enumerate(sorted(prepared, key=lambda d: d["event_rank"].iloc[0])):
            g = x[x["event_id"] == series["event_id"].iloc[0]].sort_values("datetime")
            wind_series = prep_event_series(
                g,
                value_col=wind_col,
                coverage_col="customersTracked",
                coverage_threshold=coverage_threshold,
                x_max_hours=x_max_hours,
                smooth_window=smooth_window,
                interpolate_missing=interpolate_missing,
            )
            if wind_series is None:
                continue
            color = cmap(shades[i])
            add_piecewise_line(
                ax1,
                wind_series["hour"].to_numpy(),
                wind_series["value"].to_numpy(),
                wind_series["interpolated"].to_numpy(),
                color=color,
                lw=1.2,
                alpha=0.95,
                zorder=1 + i,
            )

        ax1.set_xlim(0, x_max_hours)
        ax1.set_ylabel(wind_col)
        ax1.set_xlabel("Hours since event start")
        ax1.grid(True, alpha=0.25)

    else:
        ax0.set_xlabel("Hours since event start")

    # legend-free by default. Add one if you need it.
    fig.tight_layout()

    if outfile is not None:
        fig.savefig(outfile, bbox_inches="tight")
    return fig

In [ ]:
# Cell 5: raw outage overlay
fig = plot_overlay(
    df,
    value_col="outageFraction",
    coverage_threshold=COVERAGE_THRESHOLD,
    smooth_window=None,
    interpolate_missing=False,
    max_events=MAX_EVENTS_TO_PLOT,
    x_max_hours=X_MAX_HOURS,
    title="Raw outageFraction trajectories",
    outfile=BASE / "raw_outage_overlay.png",
)
plt.show()

In [ ]:
# Cell 6: filtered + optional interpolation + smoothing
fig = plot_overlay(
    df,
    value_col="outageFraction",
    coverage_threshold=COVERAGE_THRESHOLD,
    smooth_window=SMOOTH_WINDOW,
    interpolate_missing=USE_INTERPOLATION,
    max_events=MAX_EVENTS_TO_PLOT,
    x_max_hours=X_MAX_HOURS,
    title=f"Filtered outage trajectories | threshold={COVERAGE_THRESHOLD} | smooth={SMOOTH_WINDOW}",
    outfile=BASE / "filtered_outage_overlay.png",
)
plt.show()

In [ ]:
# Cell 7: two-panel outage + wind
fig = plot_overlay(
    df,
    value_col="outageFraction",
    wind_col="gust_mps",   # or "wind_speed_mps"
    coverage_threshold=COVERAGE_THRESHOLD,
    smooth_window=SMOOTH_WINDOW,
    interpolate_missing=USE_INTERPOLATION,
    max_events=MAX_EVENTS_TO_PLOT,
    x_max_hours=X_MAX_HOURS,
    title="Outage and wind trajectories",
    outfile=BASE / "outage_wind_overlay.png",
)
plt.show()

In [ ]:
# Cell 8: animation helper

def build_animation(
    df,
    value_col="outageFraction",
    wind_col="gust_mps",
    coverage_threshold=COVERAGE_THRESHOLD,
    smooth_window=None,
    interpolate_missing=False,
    max_events=MAX_EVENTS_TO_PLOT,
    x_max_hours=X_MAX_HOURS,
    frame_step_hours=ANIMATE_STEP_HOURS,
    outfile=BASE / "outage_wind_animation.gif",
    dpi=120,
):
    x, event_order = choose_event_subset(df, max_events=max_events)

    prepared_outage = []
    prepared_wind = []

    for eid, g in x.groupby("event_id", sort=False):
        out_s = prep_event_series(
            g,
            value_col=value_col,
            coverage_col="customersTracked",
            coverage_threshold=coverage_threshold,
            x_max_hours=x_max_hours,
            smooth_window=smooth_window,
            interpolate_missing=interpolate_missing,
        )
        if out_s is None:
            continue
        out_s["event_id"] = eid
        out_s["event_rank"] = g["event_rank"].iloc[0]
        prepared_outage.append(out_s)

        if wind_col is not None and wind_col in g.columns:
            wind_s = prep_event_series(
                g,
                value_col=wind_col,
                coverage_col="customersTracked",
                coverage_threshold=coverage_threshold,
                x_max_hours=x_max_hours,
                smooth_window=smooth_window,
                interpolate_missing=interpolate_missing,
            )
            if wind_s is not None:
                wind_s["event_id"] = eid
                wind_s["event_rank"] = g["event_rank"].iloc[0]
                prepared_wind.append(wind_s)

    if not prepared_outage:
        raise ValueError("No events available for animation.")

    # same grayscale mapping
    n = len(prepared_outage)
    shades = np.linspace(0.75, 0.05, n)
    cmap = mpl.cm.Greys

    fig, (ax0, ax1) = plt.subplots(
        2, 1, figsize=(12, 8), sharex=True,
        gridspec_kw={"height_ratios": [2, 1], "hspace": 0.08}
    )

    max_frame = int(np.floor(x_max_hours))
    frames = list(range(0, max_frame + 1, frame_step_hours))
    if frames[-1] != max_frame:
        frames.append(max_frame)

    def draw_panel(ax, series_list, frame_hour, ylabel, ylimits=None):
        ax.clear()
        for i, series in enumerate(sorted(series_list, key=lambda d: d["event_rank"].iloc[0])):
            color = cmap(shades[i])
            sub = series[series["hour"] <= frame_hour].copy()
            if sub.empty:
                continue

            # draw only the visible prefix, with interpolation style preserved
            add_piecewise_line(
                ax,
                sub["hour"].to_numpy(),
                sub["value"].to_numpy(),
                sub["interpolated"].to_numpy(),
                color=color,
                lw=1.5 if ylabel == "outageFraction" else 1.2,
                alpha=0.95,
                zorder=1 + i,
            )

        ax.set_xlim(0, x_max_hours)
        if ylimits is not None:
            ax.set_ylim(*ylimits)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)

    def update(frame_hour):
        draw_panel(ax0, prepared_outage, frame_hour, "outageFraction", (0, 1.0))
        ax0.set_title(f"Trajectory build-up through hour {frame_hour}")

        if prepared_wind:
            draw_panel(ax1, prepared_wind, frame_hour, wind_col, None)
        else:
            ax1.clear()
            ax1.set_xlim(0, x_max_hours)
            ax1.grid(True, alpha=0.25)
            ax1.set_ylabel(wind_col if wind_col else "")
        ax1.set_xlabel("Hours since event start")
        return fig.axes

    anim = FuncAnimation(fig, update, frames=frames, interval=250, blit=False, repeat=True)
    anim.save(outfile, writer=PillowWriter(fps=4), dpi=dpi)
    plt.close(fig)
    return outfile

gif_path = build_animation(
    df,
    value_col="outageFraction",
    wind_col="gust_mps",
    coverage_threshold=COVERAGE_THRESHOLD,
    smooth_window=SMOOTH_WINDOW,
    interpolate_missing=USE_INTERPOLATION,
    max_events=MAX_EVENTS_TO_PLOT,
    x_max_hours=X_MAX_HOURS,
    frame_step_hours=ANIMATE_STEP_HOURS,
    outfile=BASE / "outage_wind_trajectories.gif",
)
print(gif_path)